In [1]:
# STEP 0 — 建立分類器資料集（ROI crops，不分左右，統一方向）
from PIL import Image
import pandas as pd
from pathlib import Path

PROJ = Path(r"D:/中興大學/碩一上/仁愛醫院/仁愛醫院調整8總整理").resolve()  # ✅ 新專案根目錄
YOLO_ROOT = PROJ / "yolo_dataset_process"
IMG_ALL = YOLO_ROOT / "yolo_dataset" / "images"

df = pd.read_csv(PROJ/"roi_all.csv")  # 建議從 調整5 複製過來，或在6重跑一次生成

OUT = PROJ / "stage_cls_dataset"
OUT.mkdir(exist_ok=True, parents=True)
OUT_ROOT = Path(PROJ) / "outputs_bin(1,2,3),(4)"
# 統一成哪一邊的樣子？"L" = 全部變成左邊樣子；"R" = 全部變成右邊樣子
UNIFY_SIDE = "L"   # 老師說「都只有在左邊或右邊」，你可以先選 L

for _, r in df.iterrows():
    if pd.isna(r["x1"]):
        continue  # 沒偵測到跳過
    
    stage = int(r["stage"])
    side  = str(r["side"]).upper()  # 'L' 或 'R'
    
    out_dir = OUT / f"stage_{stage}"
    out_dir.mkdir(exist_ok=True, parents=True)
    
    img_path = IMG_ALL / r["filename"]
    x1, y1, x2, y2 = map(int, [r.x1, r.y1, r.x2, r.y2])
    
    with Image.open(img_path) as im:
        crop = im.crop((x1, y1, x2, y2))
        
        # 🔁 關鍵：把另一側翻成統一方向
        if UNIFY_SIDE == "L":
            # 希望所有關節都長得像「左側」 → 把右邊關節水平翻轉
            if side == "R":
                crop = crop.transpose(Image.FLIP_LEFT_RIGHT)
        elif UNIFY_SIDE == "R":
            # 希望都長得像「右側」 → 把左邊翻轉
            if side == "L":
                crop = crop.transpose(Image.FLIP_LEFT_RIGHT)
        else:
            raise ValueError("UNIFY_SIDE 只能是 'L' 或 'R'")
        
        crop.save(out_dir / r["filename"])

print("完成建立 stage_cls_dataset（已統一方向，不分左右）：", OUT)


完成建立 stage_cls_dataset（已統一方向，不分左右）： D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\stage_cls_dataset


In [2]:
import torch
import numpy as np
import random
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # 如果用 GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # 讓 cuDNN 的結果固定（禁用一些非確定性加速）
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("Seed fixed!")


Seed fixed!


In [3]:
from torch.utils.data import Dataset

class BinaryStageWrapper(Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        x, y = self.base[idx]      # y: 0..3
        y2 = 0 if y <= 2 else 1    # 0:stage1/2/3, 1:stage4
        return x, y2


In [4]:
# STEP 1 — 載入資料集 + train/val split
import torch
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

DATA_ROOT = OUT

# 訓練用：有增強，但❗不要再左右翻
train_tf = T.Compose([
    T.Resize((384, 384)),
    # T.RandomHorizontalFlip(),   # 先關掉，不再打亂左右
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

# 驗證用：只做必要的 resize + normalize
val_tf = T.Compose([
    T.Resize((384, 384)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

# 先只讀檔名（不指定 transform），拿到總數量
base_dataset = ImageFolder(DATA_ROOT)

print("總資料量:", len(base_dataset))
print("分類對照：", base_dataset.class_to_idx)


總資料量: 287
分類對照： {'stage_1': 0, 'stage_2': 1, 'stage_3': 2, 'stage_4': 3}


In [5]:
from pathlib import Path
import torch
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

DATA_ROOT = OUT  # 你的 stage_cls_dataset
base_dataset = ImageFolder(DATA_ROOT)  # 只用來拿 labels，不套 transform

# ===== 你想要的比例 =====
train_ratio = 0.7
val_ratio   = 0.15
test_ratio  = 0.15
assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-8

# 固定 random seed
g = torch.Generator().manual_seed(42)

# 取得每張圖的 label
labels4 = torch.tensor([y for _, y in base_dataset.samples])  # 0..3
labels2 = (labels4 == 3).long()  # 0:(stage1,2,3) 1:(stage4)

train_idx_all, val_idx_all, test_idx_all = [], [], []
num_classes = 2

for c in range(num_classes):
    idx_c = torch.where(labels2 == c)[0]
    idx_c = idx_c[torch.randperm(len(idx_c), generator=g)]

    n_c = len(idx_c)
    n_train_c = int(n_c * train_ratio)
    n_val_c   = int(n_c * val_ratio)
    n_test_c  = n_c - n_train_c - n_val_c

    train_idx_all.append(idx_c[:n_train_c])
    val_idx_all.append(idx_c[n_train_c:n_train_c + n_val_c])
    test_idx_all.append(idx_c[n_train_c + n_val_c:])

train_idx = torch.cat(train_idx_all)
val_idx   = torch.cat(val_idx_all)
test_idx  = torch.cat(test_idx_all)

# 再打散一次（可選，但我通常會做）
train_idx = train_idx[torch.randperm(len(train_idx), generator=g)]
val_idx   = val_idx[torch.randperm(len(val_idx), generator=g)]
test_idx  = test_idx[torch.randperm(len(test_idx), generator=g)]

print("train/val/test =", len(train_idx), len(val_idx), len(test_idx))

# ===== 各自有 transform 的 dataset =====
train_dataset4 = ImageFolder(DATA_ROOT, transform=train_tf)
val_dataset4   = ImageFolder(DATA_ROOT, transform=val_tf)
test_dataset4  = ImageFolder(DATA_ROOT, transform=val_tf)

train_dataset = BinaryStageWrapper(train_dataset4)
val_dataset   = BinaryStageWrapper(val_dataset4)
test_dataset  = BinaryStageWrapper(test_dataset4)


train_set = Subset(train_dataset, train_idx.tolist())
val_set   = Subset(val_dataset,   val_idx.tolist())
test_set  = Subset(test_dataset,  test_idx.tolist())

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=16, shuffle=False)

# 額外：看每個 split 的 class 分布（強烈建議）
def count_per_class(idxs):
    c = torch.bincount(labels2[idxs], minlength=2)
    return c.tolist()

print("per-class train:", count_per_class(train_idx))
print("per-class val  :", count_per_class(val_idx))
print("per-class test :", count_per_class(test_idx))



train/val/test = 200 42 45
per-class train: [154, 46]
per-class val  : [33, 9]
per-class test : [34, 11]


In [6]:
# STEP 2 — 建立分類器模型（可切換不同 backbone）
import torchvision.models as models
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

def create_model(model_name: str, num_classes: int = 2):
    """依模型名稱建立對應的預訓練模型，最後一層改成 num_classes。"""
    model_name = model_name.lower()

    if model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "efficientnet_b1":
        weights = models.EfficientNet_B1_Weights.DEFAULT
        model = models.efficientnet_b1(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif model_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT
        model = models.convnext_tiny(weights=weights)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)

    elif model_name == "convnext_small":
        weights = models.ConvNeXt_Small_Weights.DEFAULT
        model = models.convnext_small(weights=weights)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)

    elif model_name == "convnext_base":
        weights = models.ConvNeXt_Base_Weights.DEFAULT
        model = models.convnext_base(weights=weights)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)

    elif model_name == "vit_b_16":
        weights = models.ViT_B_16_Weights.DEFAULT
        model = models.vit_b_16(weights=weights)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, num_classes)

    # 🔥 新增 DenseNet（醫療分類最常用）
    elif model_name == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT
        model = models.densenet121(weights=weights)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)

    elif model_name == "densenet169":
        weights = models.DenseNet169_Weights.DEFAULT
        model = models.densenet169(weights=weights)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"未知的 model_name: {model_name}")

    return model

# 👉 在這裡選你要的 backbone
# model_name = "convnext_tiny"   # 改成 "efficientnet_b1" / "resnet50" / "convnext_tiny" / "vit_b_16"

# model = create_model(model_name, num_classes=4).to(device)
# print("使用模型:", model_name)


device = cuda


In [7]:
# STEP 3 — train_one_model（+ train_log.csv + training time）
import torch
import torch.optim as optim
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
import time
import pandas as pd

def train_one_model(
    model_name,
    create_model_fn,
    train_loader,
    val_loader,
    device,
    PROJ,
    num_classes=2,
    epochs=30,
    lr=1e-4,
):
    out_dir = Path(PROJ) / "outputs_bin(1,2,3),(4)" / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path = out_dir / f"best_{model_name}.pth"
    log_path  = out_dir / "train_log.csv"

    model = create_model_fn(model_name, num_classes=num_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # ✅ best 改成看 val_macro_f1
    best_val_macro_f1 = -1.0
    best_epoch = -1

    # 🔹 收集 epoch log
    logs = []

    t_start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(imgs)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += imgs.size(0)

        train_loss = total_loss / max(total, 1)
        train_acc  = correct / max(total, 1)

        # ---- validation ----
        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                labels = labels.to(device)
                logits = model(imgs)
                pred = logits.argmax(dim=1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(pred.cpu().numpy())

        val_acc = accuracy_score(y_true, y_pred)
        val_macro_f1 = f1_score(y_true, y_pred, average="macro")

        print(
            f"[{model_name}] Epoch {epoch:02d}/{epochs} | "
            f"loss={train_loss:.4f} acc={train_acc:.3f} "
            f"val_acc={val_acc:.3f} val_macro_f1={val_macro_f1:.3f}"
        )

        # 🔹 存 log（多存一欄 val_macro_f1）
        logs.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "val_macro_f1": val_macro_f1,
        })

        # 🔥 best model（改用 val_macro_f1）
        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)
            print(f"  👉 New best saved (epoch={epoch}, val_macro_f1={best_val_macro_f1:.3f})")

    t_end = time.time()
    train_time_sec = t_end - t_start

    # 🔹 儲存 train_log.csv
    df_log = pd.DataFrame(logs)
    df_log.to_csv(log_path, index=False, encoding="utf-8-sig")

    print(f"📄 saved train log -> {log_path}")
    print(f"⏱ training time = {train_time_sec:.1f} sec ({train_time_sec/60:.2f} min)")

    # load best model
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    # ✅ 第二個回傳值現在是 best_val_macro_f1（位置不變）
    return model, best_val_macro_f1, best_epoch, ckpt_path, train_time_sec



In [8]:
# STEP 4 — eval + save：自動存 confusion matrix + report 到 outputs/{model_name}/
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

def eval_and_save(model, loader, class_names, model_name, split_name, device, PROJ):
    out_dir = Path(PROJ) / "outputs_bin(1,2,3),(4)" / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    y_true, y_pred = [], []

    model.eval()
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            logits = model(imgs)
            pred = logits.argmax(dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    cm = confusion_matrix(y_true, y_pred)

    # --- save confusion matrix figure ---
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name} | {split_name}")

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(
                j, i, cm[i, j],
                ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black"
            )

    plt.colorbar(im, ax=ax)
    plt.tight_layout()

    cm_path = out_dir / f"confusion_matrix_{split_name}.png"
    plt.savefig(cm_path, dpi=300, bbox_inches="tight")
    plt.close()

    # --- save classification report ---
    report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    report_path = out_dir / f"classification_report_{split_name}.txt"
    report_path.write_text(report, encoding="utf-8")

    return {
        "acc": float(acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "cm_path": str(cm_path),
        "report_path": str(report_path),
    }


In [9]:
# STEP 5 — 一鍵 train 所有模型 + 自動輸出 outputs/summary.csv + 自動挑 best 並複製到 outputs/_best_model
import pandas as pd
import torch
from pathlib import Path
import shutil

ALL_MODELS = [
    "efficientnet_b0",
    "efficientnet_b1",
    "resnet50",
    "convnext_tiny",
    "convnext_small",
    "densenet121",
    "densenet169",
    # "vit_b_16",
]

class_names = ["stage_1-2-3", "stage_4"]

summaries = []

for mn in ALL_MODELS:
    try:
        print("\n" + "="*100)
        print("TRAIN MODEL:", mn)

        # train_one_model 請確保回傳：
        # model, best_val_acc, best_epoch, ckpt_path, train_time_sec
        model, best_val_macro_f1, best_epoch, ckpt_path, train_time_sec = train_one_model(
            model_name=mn,
            create_model_fn=create_model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            PROJ=PROJ,
            num_classes=2,
            epochs=30,
            lr=1e-4,
        )

        # eval + save val/test（請確保 eval_and_save 有回傳 acc/macro_f1/weighted_f1/cm_path/report_path）
        val_metrics  = eval_and_save(model, val_loader,  class_names, mn, "val",  device, PROJ)
        test_metrics = eval_and_save(model, test_loader, class_names, mn, "test", device, PROJ)

        summaries.append({
            "model_name": mn,
            "best_epoch": int(best_epoch),
            "best_ckpt": str(ckpt_path),
            "best_val_macro_f1_during_train": float(best_val_macro_f1),

            "train_time_sec": round(float(train_time_sec), 1),
            "train_time_min": round(float(train_time_sec) / 60, 2),

            "val_acc": float(val_metrics["acc"]),
            "val_macro_f1": float(val_metrics["macro_f1"]),
            "val_weighted_f1": float(val_metrics["weighted_f1"]),

            "test_acc": float(test_metrics["acc"]),
            "test_macro_f1": float(test_metrics["macro_f1"]),
            "test_weighted_f1": float(test_metrics["weighted_f1"]),

            "val_cm_path": str(val_metrics["cm_path"]),
            "test_cm_path": str(test_metrics["cm_path"]),
            "val_report_path": str(val_metrics.get("report_path", "")),
            "test_report_path": str(test_metrics.get("report_path", "")),
        })

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"[!] Model {mn} failed:", repr(e))
        summaries.append({"model_name": mn, "error": repr(e)})

# ===== Summary DataFrame =====
df_summary = pd.DataFrame(summaries)

# 只挑沒有 error 的列來選 best
df_ok = df_summary.copy()
if "error" in df_ok.columns:
    df_ok = df_ok[df_ok["error"].isna()]

# ===== Select BEST model by val_macro_f1 =====

# 預設沒有 best
df_summary["is_best_val_macro_f1"] = False

best_model = None
best_row = None

# 只挑沒有 error 的列
df_ok = df_summary.copy()
if "error" in df_ok.columns:
    df_ok = df_ok[df_ok["error"].isna()]

if len(df_ok) == 0:
    print("[!] No successful models to select best from.")
else:
    if "val_macro_f1" not in df_ok.columns:
        print("[!] val_macro_f1 not found in summary, cannot select best.")
    else:
        # 👉 用 val_macro_f1 選最佳
        best_row = df_ok.sort_values(by="val_macro_f1", ascending=False).iloc[0]
        best_model = best_row["model_name"]

        # 標記 best
        df_summary["is_best_val_macro_f1"] = (
            df_summary["model_name"] == best_model
        )

        print(
            f"🏆 BEST model by val_macro_f1 = {best_model} "
            f"(val_macro_f1={best_row['val_macro_f1']:.4f})"
        )

# 排序（讓表格好看）
if "val_macro_f1" in df_summary.columns:
    df_summary = df_summary.sort_values(by="val_macro_f1", ascending=False)


# ===== 寫出 summary.csv =====
summary_path = OUT_ROOT / "summary.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
print("\n✅ saved summary:", summary_path)

# ===== 複製 best model artifacts 到 outputs/_best_model =====
best_dir = OUT_ROOT / "_best_model"
best_dir.mkdir(parents=True, exist_ok=True)

if best_model is None:
    print("[!] Best model not selected, skip copying.")
else:
    # 清空舊 best_dir（可選）
    for p in best_dir.glob("*"):
        if p.is_file():
            p.unlink()
        elif p.is_dir():
            shutil.rmtree(p)

    src_dir = OUT_ROOT / best_model

    keep_files = [
        f"best_{best_model}.pth",
        "train_log.csv",
        "confusion_matrix_val.png",
        "confusion_matrix_test.png",
        "classification_report_val.txt",
        "classification_report_test.txt",
    ]

    for fn in keep_files:
        src = src_dir / fn
        if src.exists():
            shutil.copy2(src, best_dir / fn)

    # BEST_INFO.txt
    (best_dir / "BEST_INFO.txt").write_text(
        f"Best model selected by test_macro_f1\n"
        f"model_name: {best_model}\n"
        f"test_macro_f1: {float(best_row['test_macro_f1']):.6f}\n"
        f"test_acc: {float(best_row.get('test_acc', float('nan'))):.6f}\n"
        f"val_macro_f1: {float(best_row.get('val_macro_f1', float('nan'))):.6f}\n"
        f"val_acc: {float(best_row.get('val_acc', float('nan'))):.6f}\n",
        encoding="utf-8"
    )

    print("✅ Copied best artifacts to:", best_dir)

df_summary




TRAIN MODEL: efficientnet_b0
[efficientnet_b0] Epoch 01/30 | loss=0.6059 acc=0.725 val_acc=0.452 val_macro_f1=0.437
  👉 New best saved (epoch=1, val_macro_f1=0.437)
[efficientnet_b0] Epoch 02/30 | loss=0.4546 acc=0.805 val_acc=0.714 val_macro_f1=0.630
  👉 New best saved (epoch=2, val_macro_f1=0.630)
[efficientnet_b0] Epoch 03/30 | loss=0.3800 acc=0.840 val_acc=0.786 val_macro_f1=0.694
  👉 New best saved (epoch=3, val_macro_f1=0.694)
[efficientnet_b0] Epoch 04/30 | loss=0.3095 acc=0.880 val_acc=0.810 val_macro_f1=0.717
  👉 New best saved (epoch=4, val_macro_f1=0.717)
[efficientnet_b0] Epoch 05/30 | loss=0.2431 acc=0.930 val_acc=0.905 val_macro_f1=0.869
  👉 New best saved (epoch=5, val_macro_f1=0.869)
[efficientnet_b0] Epoch 06/30 | loss=0.1485 acc=0.990 val_acc=0.881 val_macro_f1=0.841
[efficientnet_b0] Epoch 07/30 | loss=0.1234 acc=0.965 val_acc=0.881 val_macro_f1=0.797
[efficientnet_b0] Epoch 08/30 | loss=0.1658 acc=0.970 val_acc=0.905 val_macro_f1=0.846
[efficientnet_b0] Epoch 09/30

,model_name,best_epoch,best_ckpt,best_val_macro_f1_during_train,train_time_sec,train_time_min,val_acc,val_macro_f1,val_weighted_f1,test_acc,test_macro_f1,test_weighted_f1,val_cm_path,test_cm_path,val_report_path,test_report_path,is_best_val_macro_f1
5,densenet121,27,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",0.963126,70.1,1.17,0.976190,0.963126,0.975668,0.977778,0.968944,0.977410,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",True
4,convnext_small,14,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",0.963126,181.1,3.02,0.976190,0.963126,0.975668,0.933333,0.906832,0.932229,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",False
3,convnext_tiny,7,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",0.963126,120.7,2.01,0.976190,0.963126,0.975668,0.977778,0.970798,0.978095,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",False
0,efficientnet_b0,15,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",0.929293,46.4,0.77,0.952381,0.929293,0.952381,0.955556,0.939840,0.955556,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",False
1,efficientnet_b1,17,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",0.929293,56.0,0.93,0.952381,0.929293,0.952381,0.911111,0.879679,0.911111,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",False
2,resnet50,22,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",0.922794,64.9,1.08,0.952381,0.922794,0.950105,0.933333,0.912395,0.934285,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",False
6,densenet169,21,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",0.922794,81.0,1.35,0.952381,0.922794,0.950105,0.955556,0.935714,0.953968,"D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...","D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3)...",False


In [ ]:
# STEP 6 — Grad-CAM 工具

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import cv2

# ==========================================
# 1. 基礎設定與函式
# ==========================================

# ⚠️ 根據你訓練時的 Normalize 設定
MEAN = [0.5, 0.5, 0.5]
STD  = [0.5, 0.5, 0.5]
device = "cuda" if torch.cuda.is_available() else "cpu"

def denormalize(img_tensor):
    """反正規化：Tensor -> Numpy Image [0,1]"""
    img = img_tensor.clone().cpu().numpy()
    for c in range(3):
        img[c] = img[c] * STD[c] + MEAN[c]
    img = np.clip(img, 0, 1)
    img = np.transpose(img, (1, 2, 0))
    return img

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, inp, out):
            self.activations = out.detach()
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def __call__(self, x, class_idx=None):
        self.model.zero_grad()
        out = self.model(x)
        if class_idx is None:
            class_idx = out.argmax(dim=1).item()
        score = out[0, class_idx]
        score.backward()
        
        acts = self.activations
        grads = self.gradients
        weights = grads.mean(dim=(2, 3))[0]
        
        gcam = torch.zeros(acts.shape[2:], dtype=torch.float32).to(acts.device)
        for i, w in enumerate(weights):
            gcam += w * acts[0, i, :, :]
            
        gcam = F.relu(gcam)
        gcam -= gcam.min()
        if gcam.max() > 0:
            gcam /= gcam.max()
        return gcam.detach().cpu().numpy()

def get_target_layer(model, model_name: str):
    model_name = model_name.lower()
    if model_name in ["convnext_tiny", "convnext_small", "convnext_base"]:
        return model.features[-1]
    elif model_name == "resnet50":
        return model.layer4
    elif model_name in ["densenet121", "densenet169"]:
        # DenseNet 的 features 包含所有 DenseBlock，通常取最後一層輸出
        return model.features
    elif model_name in ["efficientnet_b0", "efficientnet_b1"]:
        return model.features[-1]
    else:
        raise ValueError(f"Unknown model name: {model_name}")

In [11]:
# ==========================================
# 2. 🔥 關鍵：載入最佳模型 (避免用到最後訓練的那一個)
# ==========================================

# 從上一部 Summary 找出最佳模型名稱 (或是你自己手動指定，例如 "densenet121")
# 假設 best_model 變數還在記憶體中，如果不在，請手動改成字串，如 BEST_MODEL_NAME = "densenet121"
try:
    BEST_MODEL_NAME = best_model 
except NameError:
    BEST_MODEL_NAME = "densenet121"  # ⚠️ 如果報錯，請手動改成你剛剛跑出來的第一名模型名稱

print(f"🔥 正在載入最佳模型：{BEST_MODEL_NAME} ...")

# 重新建立架構
model_vis = create_model(BEST_MODEL_NAME, num_classes=2).to(device)

# 載入權重 (從 outputs/_best_model 讀取)
best_ckpt_path = PROJ / "outputs_bin(1,2,3),(4)" / "_best_model" / f"best_{BEST_MODEL_NAME}.pth"
if best_ckpt_path.exists():
    model_vis.load_state_dict(torch.load(best_ckpt_path, map_location=device))
    print(f"✅ 成功載入權重：{best_ckpt_path}")
else:
    print(f"❌ 找不到權重檔：{best_ckpt_path}，請檢查路徑！")

🔥 正在載入最佳模型：densenet121 ...
✅ 成功載入權重：D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\outputs_bin(1,2,3),(4)\_best_model\best_densenet121.pth


In [12]:
# ==========================================
# 3. 執行 Grad-CAM 並存檔
# ==========================================

target_layer = get_target_layer(model_vis, BEST_MODEL_NAME)
gcam = GradCAM(model_vis, target_layer)

# 動態設定資料夾名稱
GCAM_DIR = PROJ / f"m1_gradcam_{BEST_MODEL_NAME}"
GCAM_DIR.mkdir(exist_ok=True, parents=True)
print("📂 Grad-CAM 圖片將存到：", GCAM_DIR)

n_show = 45  # 設定要輸出的張數
shown = 0

# 定義顯示用的類別名稱 (對應 0 和 1)
class_map = {0: "Stage 1-3", 1: "Stage 4"}

model_vis.eval()

# 這裡不加 no_grad，因為 Grad-CAM 需要 backward 算梯度
for imgs, labels in test_loader:
    imgs = imgs.to(device)
    labels = labels.to(device)

    batch_size = imgs.size(0)
    for i in range(batch_size):
        if shown >= n_show:
            break

        img = imgs[i:i+1]
        label_idx = labels[i].item()
        label_str = class_map[label_idx] # 轉成文字 Stage 1-3 或 Stage 4

        # Forward
        model_vis.zero_grad()
        out = model_vis(img)
        pred_idx = out.argmax(1).item()
        pred_str = class_map[pred_idx]

        # 計算 Grad-CAM
        gcam_map = gcam(img, class_idx=pred_idx)

        # 畫圖
        img_np = denormalize(img[0])
        h, w, _ = img_np.shape
        gcam_resized = cv2.resize(gcam_map, (w, h))
        heatmap = cv2.applyColorMap(np.uint8(255 * gcam_resized), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
        overlay = np.clip(0.4 * heatmap + 0.6 * img_np, 0, 1)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        
        axes[0].imshow(img_np)
        axes[0].set_title(f"Original\nGT: {label_str}")
        axes[0].axis("off")

        axes[1].imshow(gcam_resized, cmap="jet")
        axes[1].set_title(f"Grad-CAM\nPred: {pred_str}")
        axes[1].axis("off")

        axes[2].imshow(overlay)
        axes[2].set_title("Overlay")
        axes[2].axis("off")

        # 檔名加入 GT 和 Pred 資訊方便篩選
        file_name = f"m1_test_{shown+1}_GT-{label_idx}_Pred-{pred_idx}.png"
        save_path = GCAM_DIR / file_name
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig) # 關閉圖片釋放記憶體，避免大量迴圈時爆掉

        if shown % 5 == 0:
            print(f"Saved: {file_name}")
            
        shown += 1

    if shown >= n_show:
        break

print(f"\n✅ 全部完成！請到資料夾查看圖片：\n{GCAM_DIR}")

📂 Grad-CAM 圖片將存到： D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\m1_gradcam_densenet121


c:\Users\USER\anaconda3\envs\unet_labeling\Lib\site-packages\torch\nn\modules\module.py:1868: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Saved: m1_test_1_GT-1_Pred-1.png
Saved: m1_test_6_GT-0_Pred-0.png
Saved: m1_test_11_GT-0_Pred-0.png
Saved: m1_test_16_GT-1_Pred-1.png
Saved: m1_test_21_GT-0_Pred-0.png
Saved: m1_test_26_GT-0_Pred-0.png
Saved: m1_test_31_GT-1_Pred-0.png
Saved: m1_test_36_GT-0_Pred-0.png
Saved: m1_test_41_GT-0_Pred-0.png

✅ 全部完成！請到資料夾查看圖片：
D:\中興大學\碩一上\仁愛醫院\仁愛醫院調整8總整理\m1_gradcam_densenet121
